In [1]:
from __future__ import annotations

import random
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

PROJECT_ROOT = Path.cwd()
CELEBA_ROOT = PROJECT_ROOT / "celebadata"

ATTR_CSV_PATH = CELEBA_ROOT / "list_attr_celeba.csv"
PARTITION_CSV_PATH = CELEBA_ROOT / "list_eval_partition.csv"

PREPARED_ROOT = PROJECT_ROOT / "celebadata_prepared"
PREPARED_IMAGES_DIR = PREPARED_ROOT / "images_128_mtcnn"

IMAGE_SIZE = 128
BATCH_SIZE = 64
NUM_WORKERS = 4
MAX_SAMPLES = 124000

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)


DEVICE: cuda


In [4]:
attr_df = pd.read_csv(ATTR_CSV_PATH)
part_df = pd.read_csv(PARTITION_CSV_PATH)

prepared_df = part_df.merge(attr_df[["image_id", "Male"]], on="image_id", how="inner")

split_map = {0: "train", 1: "val", 2: "test"}
prepared_df["split_name"] = prepared_df["partition"].map(split_map)
prepared_df["prepared_path"] = prepared_df["image_id"].apply(lambda x: str(PREPARED_IMAGES_DIR / x))
prepared_df["male"] = (prepared_df["Male"] == 1).astype(int)

# оставляем только реально подготовленные изображения
prepared_df = prepared_df[prepared_df["prepared_path"].map(lambda p: Path(p).exists())].reset_index(drop=True)

# если val/test пустые -> делим train в пропорции 80/10/10
vc = prepared_df["split_name"].value_counts()
if vc.get("val", 0) == 0 and vc.get("test", 0) == 0 and vc.get("train", 0) > 0:
    from sklearn.model_selection import train_test_split

    tr = prepared_df[prepared_df["split_name"] == "train"].copy()

    tr_val, te = train_test_split(
        tr, test_size=0.10, random_state=SEED, stratify=tr["male"]
    )
    tr_new, va = train_test_split(
        tr_val, test_size=(0.10 / 0.90), random_state=SEED, stratify=tr_val["male"]
    )

    tr_new["split_name"] = "train"
    va["split_name"] = "val"
    te["split_name"] = "test"

    prepared_df = pd.concat([tr_new, va, te], ignore_index=True)

# ограничение до MAX_SAMPLES с сохранением пропорций
if len(prepared_df) > MAX_SAMPLES:
    part = prepared_df["split_name"].value_counts(normalize=True)
    target_counts = (part * MAX_SAMPLES).round().astype(int)

    diff = MAX_SAMPLES - target_counts.sum()
    if diff != 0:
        target_counts.iloc[0] += diff

    chunks = []
    for split_name, n in target_counts.items():
        split_df = prepared_df[prepared_df["split_name"] == split_name]
        chunks.append(split_df.sample(n=min(n, len(split_df)), random_state=SEED))
    prepared_df = pd.concat(chunks, ignore_index=True)

prepared_df = prepared_df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
prepared_df = prepared_df[["prepared_path", "split_name", "male"]]

print(prepared_df["split_name"].value_counts())
print("Total:", len(prepared_df))


split_name
train    95368
test     11922
val      11922
Name: count, dtype: int64
Total: 119212


In [ ]:
part_df = pd.read_csv(PARTITION_CSV_PATH)
print(part_df["partition"].value_counts())

tmp = part_df.merge(attr_df[["image_id","Male"]], on="image_id")
tmp["split_name"] = tmp["partition"].map({0:"train",1:"val",2:"test"})
tmp["prepared_path"] = tmp["image_id"].apply(lambda x: str(PREPARED_IMAGES_DIR / x))
tmp["exists"] = tmp["prepared_path"].map(lambda p: Path(p).exists())
print(tmp.groupby("split_name")["exists"].sum())


NameError: name 'meta_df' is not defined

In [50]:

class CelebaPreparedDataset(Dataset):
    def __init__(self, df: pd.DataFrame, split_name: str):
        self.df = df[df["split_name"] == split_name].reset_index(drop=True).copy()
        self.tfm = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
        ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        x = self.tfm(Image.open(row["prepared_path"]).convert("RGB"))
        y = int(row["male"])
        return x, y

train_ds = CelebaPreparedDataset(prepared_df, "train")
val_ds = CelebaPreparedDataset(prepared_df, "val")
test_ds = CelebaPreparedDataset(prepared_df, "test")

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

print(len(train_ds), len(val_ds), len(test_ds))


95368 11922 11922


In [ ]:
torch.autograd.set_detect_anomaly(True, check_nan=False)



In [ ]:
def prepare_faces(df: pd.DataFrame, max_images: int | None = None, image_size: int = 128, min_conf: float = 0.90):
    detector = MTCNN(image_size=image_size, margin=18, post_process=False, device=DEVICE)
    rows = []
    work_df = df if max_images is None else df.iloc[:max_images].copy()

    for _, row in tqdm(work_df.iterrows(), total=len(work_df), desc="Preparing faces"):
        image_id = row["image_id"]
        src = RAW_IMAGES_DIR / image_id
        dst = PREPARED_IMAGES_DIR / image_id

        if not dst.exists():
            img = Image.open(src).convert("RGB")
            boxes, probs = detector.detect(img)
            ok = 0

            if boxes is not None and probs is not None and len(boxes) > 0:
                i = int(np.argmax(probs))
                if probs[i] >= min_conf:
                    x1, y1, x2, y2 = boxes[i].astype(int).tolist()
                    x1, y1 = max(0, x1), max(0, y1)
                    x2, y2 = min(img.width, x2), min(img.height, y2)
                    if (x2 - x1) > 20 and (y2 - y1) > 20:
                        face = img.crop((x1, y1, x2, y2)).resize((image_size, image_size), Image.BILINEAR)
                        face.save(dst, quality=95)
                        ok = 1

            if ok == 0:
                img.resize((image_size, image_size), Image.BILINEAR).save(dst, quality=95)
        else:
            ok = 1

        rows.append({
            "image_id": image_id,
            "prepared_path": str(dst),
            "split": int(row["split"]),
            "split_name": row["split_name"],
            "male": int(row["Male"]),
            "ok": int(ok),
        })

    prepared_df = pd.DataFrame(rows)
    prepared_df.to_csv(PREPARED_META_PATH, index=False)
    return prepared_df

meta_df_chunk = meta_df.iloc[53_000:120_000].copy()

prepared_df = prepare_faces(
    meta_df_chunk,
    max_images=None,
    image_size=IMAGE_SIZE
)

display(prepared_df.head())
print("Rows:", len(prepared_df))
print("Face detected ratio:", prepared_df["ok"].mean())
print(prepared_df["split_name"].value_counts())




Preparing faces:   0%|          | 0/67000 [00:00<?, ?it/s]

,image_id,prepared_path,split,split_name,male,ok
0,053001.jpg,c:\Users\wiad_\PycharmProjects\ItmoCv\celebada...,0,train,1,1
1,053002.jpg,c:\Users\wiad_\PycharmProjects\ItmoCv\celebada...,0,train,0,1
2,053003.jpg,c:\Users\wiad_\PycharmProjects\ItmoCv\celebada...,0,train,0,1
3,053004.jpg,c:\Users\wiad_\PycharmProjects\ItmoCv\celebada...,0,train,0,1
4,053005.jpg,c:\Users\wiad_\PycharmProjects\ItmoCv\celebada...,0,train,0,1


Rows: 67000
Face detected ratio: 0.9974328358208955
split_name
train    67000
Name: count, dtype: int64


In [ ]:
def show_batch(loader, n=16, title="batch"):
    x, _ = next(iter(loader))
    x = x[:n]
    grid = make_grid((x * 0.5 + 0.5).clamp(0, 1), nrow=int(math.sqrt(n)))
    plt.figure(figsize=(7, 7))
    plt.title(title)
    plt.imshow(grid.permute(1, 2, 0).cpu().numpy())
    plt.axis("off")
    plt.show()

show_batch(train_loader, n=16, title="Prepared CelebA")


In [ ]:
def weights_init(m):
    name = m.__class__.__name__
    if "Conv" in name or "Linear" in name:
        if getattr(m, "weight", None) is not None:
            nn.init.normal_(m.weight.data, 0.0, 0.02)
        if getattr(m, "bias", None) is not None:
            nn.init.constant_(m.bias.data, 0.0)
    elif "BatchNorm" in name:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0.0)

class ConditionalGenerator(nn.Module):
    def __init__(self, z_dim=128, n_cls=2, emb_dim=16):
        super().__init__()
        self.emb = nn.Embedding(n_cls, emb_dim)
        self.backbone = Generator(z_dim + emb_dim).net

    def forward(self, z, y):
        y = y.view(-1).long()
        assert z.size(0) == y.size(0), f"G: z={z.size(0)} y={y.size(0)}"
        e = self.emb(y).unsqueeze(-1).unsqueeze(-1)
        return self.backbone(torch.cat([z, e], dim=1))

class Generator(nn.Module):
    def __init__(self, z_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.ConvTranspose2d(z_dim, 512, 4, 1, 0, bias=False),
            nn.BatchNorm2d(512), nn.ReLU(True),
            nn.ConvTranspose2d(512, 256, 4, 2, 1, bias=False),
            nn.BatchNorm2d(256), nn.ReLU(True),
            nn.ConvTranspose2d(256, 128, 4, 2, 1, bias=False),
            nn.BatchNorm2d(128), nn.ReLU(True),
            nn.ConvTranspose2d(128, 64, 4, 2, 1, bias=False),
            nn.BatchNorm2d(64), nn.ReLU(True),
            nn.ConvTranspose2d(64, 3, 4, 2, 1, bias=False),
            nn.Tanh(),
        )

    def forward(self, z):
        return self.net(z)


class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 64, 4, 2, 1, bias=False), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(64, 128, 4, 2, 1, bias=False), nn.BatchNorm2d(128), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(128, 256, 4, 2, 1, bias=False), nn.BatchNorm2d(256), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(256, 512, 4, 2, 1, bias=False), nn.BatchNorm2d(512), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(512, 1, 4, 1, 0, bias=False), nn.Sigmoid(),
        )

    def forward(self, x):
        return self.net(x).view(-1)


class ConditionalDiscriminator(nn.Module):
    def __init__(self, n_cls=2, hw=128):
        super().__init__()
        self.emb = nn.Embedding(n_cls, hw * hw)
        self.first = nn.Sequential(
            nn.Conv2d(4, 64, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
        )
        tail = list(Discriminator().net.children())[1:]
        self.tail = nn.Sequential(*tail)

    def forward(self, x, y):
        y = y.view(-1).long()                  # важно
        assert x.size(0) == y.size(0), f"D: x={x.size(0)} y={y.size(0)}"
        ymap = self.emb(y).view(-1, 1, x.size(2), x.size(3))
        h = self.first(torch.cat([x, ymap], dim=1))
        out = self.tail(h)
        if out.ndim == 4:
            out = out.mean(dim=(2, 3))
        return out.view(-1)


class ConditionalDiscriminator(nn.Module):
    def __init__(self, n_cls=2):
        super().__init__()
        self.emb = nn.Embedding(n_cls, 1)  # 1 scalar per class
        self.first = nn.Sequential(
            nn.Conv2d(4, 64, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=False),
        )
        tail = list(Discriminator().net.children())[1:]
        self.tail = nn.Sequential(*tail)

    def forward(self, x, y):
        y = y.view(-1).long()
        assert x.size(0) == y.size(0), f"x batch={x.size(0)}, y batch={y.size(0)}"

        ymap = self.emb(y).view(-1, 1, 1, 1).expand(-1, 1, x.size(2), x.size(3))
        h = self.first(torch.cat([x, ymap], dim=1))
        out = self.tail(h)
        if out.ndim == 4:
            out = out.mean(dim=(2, 3))
        return out.view(-1)



def train_cgan(G, D, loader, epochs=5):
    G, D = G.to(DEVICE), D.to(DEVICE)
    G.apply(weights_init)
    D.apply(weights_init)

    opt_g = torch.optim.Adam(G.parameters(), lr=LR, betas=BETAS)
    opt_d = torch.optim.Adam(D.parameters(), lr=LR, betas=BETAS)
    bce = nn.BCELoss()

    hist = {"g_loss": [], "d_loss": []}

    for ep in range(1, epochs + 1):
        g_losses, d_losses = [], []

        for real, y in tqdm(loader, desc=f"cGAN {ep}/{epochs}"):
            real, y = real.to(DEVICE), y.to(DEVICE)
            bsz = real.size(0)

            opt_d.zero_grad(set_to_none=True)

            pred_real = D(real, y)
            loss_real = bce(pred_real, torch.ones_like(pred_real))

            z = torch.randn(bsz, Z_DIM, 1, 1, device=DEVICE)
            fake = G(z, y).detach()
            pred_fake = D(fake, y)
            loss_fake = bce(pred_fake, torch.zeros_like(pred_fake))

            d_loss = loss_real + loss_fake
            d_loss.backward()
            opt_d.step()

            opt_g.zero_grad(set_to_none=True)

            z = torch.randn(bsz, Z_DIM, 1, 1, device=DEVICE)
            fake = G(z, y)
            pred_fake_g = D(fake, y)
            g_loss = bce(pred_fake_g, torch.ones_like(pred_fake_g))

            g_loss.backward()
            opt_g.step()

            g_losses.append(g_loss.item())
            d_losses.append(d_loss.item())

        hist["g_loss"].append(float(np.mean(g_losses)))
        hist["d_loss"].append(float(np.mean(d_losses)))

    return G, D, hist


real, y = next(iter(train_loader))
real, y = real.to(DEVICE), y.to(DEVICE).view(-1).long()

pred = ConditionalDiscriminator().to(DEVICE)(real, y)
print("pred shape:", pred.shape, "bsz:", real.size(0))

cgan_G, cgan_D, cgan_hist = train_cgan(
    ConditionalGenerator(Z_DIM),
    ConditionalDiscriminator(),
    train_loader,
    epochs=10
)


pred shape: torch.Size([64]) bsz: 64


cGAN 1/10:   0%|          | 0/1490 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
@torch.no_grad()
def show_same_z_diff_labels(G, n=8):
    G.eval()
    z = torch.randn(n, Z_DIM, 1, 1, device=DEVICE)
    y0 = torch.zeros(n, dtype=torch.long, device=DEVICE)
    y1 = torch.ones(n, dtype=torch.long, device=DEVICE)

    x0 = G(z, y0)
    x1 = G(z, y1)
    both = torch.cat([x0, x1], dim=0)
    grid = make_grid((both * 0.5 + 0.5).clamp(0, 1).cpu(), nrow=n)

    plt.figure(figsize=(2*n, 4))
    plt.title("Top: label=0, Bottom: label=1 (same z)")
    plt.imshow(grid.permute(1, 2, 0).numpy())
    plt.axis("off")
    plt.show()

show_same_z_diff_labels(cgan_G, n=8)


In [ ]:
@torch.no_grad()
def collect_real_u8(loader, n=2048):
    xs = []
    for x, _ in loader:
        xs.append(x)
        if sum(t.size(0) for t in xs) >= n:
            break
    x = torch.cat(xs, dim=0)[:n]
    return ((x * 0.5 + 0.5).clamp(0, 1) * 255).to(torch.uint8)

@torch.no_grad()
def collect_fake_u8(G, n=2048, conditional=False, label=0):
    G.eval()
    out = []
    while sum(t.size(0) for t in out) < n:
        b = min(256, n - sum(t.size(0) for t in out))
        z = torch.randn(b, Z_DIM, 1, 1, device=DEVICE)
        if conditional:
            y = torch.full((b,), label, dtype=torch.long, device=DEVICE)
            x = G(z, y)
        else:
            x = G(z)
        out.append(((x * 0.5 + 0.5).clamp(0, 1) * 255).to(torch.uint8).cpu())
    return torch.cat(out, dim=0)[:n]

def compute_fid_is(real_u8, fake_u8):
    fid_metric = FrechetInceptionDistance(feature=2048, normalize=False).to(DEVICE)
    is_metric = InceptionScore(normalize=False).to(DEVICE)

    fid_metric.update(real_u8.to(DEVICE), real=True)
    fid_metric.update(fake_u8.to(DEVICE), real=False)
    fid = float(fid_metric.compute().item())

    is_metric.update(fake_u8.to(DEVICE))
    is_mean, is_std = is_metric.compute()
    return {"FID": fid, "IS_mean": float(is_mean.item()), "IS_std": float(is_std.item())}

real_u8 = collect_real_u8(test_loader, n=2048)
gan_u8 = collect_fake_u8(gan_G, n=2048, conditional=False)
cgan_u8 = collect_fake_u8(cgan_G, n=2048, conditional=True, label=1)

gan_metrics = compute_fid_is(real_u8, gan_u8)
cgan_metrics = compute_fid_is(real_u8, cgan_u8)
print("GAN:", gan_metrics)
print("cGAN:", cgan_metrics)


In [45]:
class Critic(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 64, 4, 2, 1), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(64, 128, 4, 2, 1), nn.InstanceNorm2d(128, affine=True), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(128, 256, 4, 2, 1), nn.InstanceNorm2d(256, affine=True), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(256, 512, 4, 2, 1), nn.InstanceNorm2d(512, affine=True), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(512, 1, 4, 1, 0),
        )

    def forward(self, x):
        return self.net(x).view(-1)

def gradient_penalty(C, real, fake):
    alpha = torch.rand(real.size(0), 1, 1, 1, device=DEVICE)
    mix = (alpha * real + (1 - alpha) * fake).requires_grad_(True)
    score = C(mix)
    grad = torch.autograd.grad(
        outputs=score, inputs=mix, grad_outputs=torch.ones_like(score),
        create_graph=True, retain_graph=True, only_inputs=True
    )[0]
    grad = grad.view(grad.size(0), -1)
    return ((grad.norm(2, dim=1) - 1.0) ** 2).mean()

def train_wgan_gp(G, C, loader, epochs=5, n_critic=5, gp_lambda=10.0):
    G, C = G.to(DEVICE), C.to(DEVICE)
    G.apply(weights_init); C.apply(weights_init)
    opt_g = torch.optim.Adam(G.parameters(), lr=1e-4, betas=(0.0, 0.9))
    opt_c = torch.optim.Adam(C.parameters(), lr=1e-4, betas=(0.0, 0.9))
    hist = {"g_loss": [], "c_loss": []}

    for ep in range(1, epochs + 1):
        g_losses, c_losses = [], []
        for real, _ in tqdm(loader, desc=f"WGAN-GP {ep}/{epochs}"):
            real = real.to(DEVICE, non_blocking=True)
            
            bsz = real.size(0)

            for _ in range(n_critic):
                z = torch.randn(bsz, Z_DIM, 1, 1, device=DEVICE)
                fake = G(z).detach()
                c_real = C(real).mean()
                c_fake = C(fake).mean()
                gp = gradient_penalty(C, real, fake)
                c_loss = -(c_real - c_fake) + gp_lambda * gp

                opt_c.zero_grad(set_to_none=True)
                c_loss.backward()
                opt_c.step()

            z = torch.randn(bsz, Z_DIM, 1, 1, device=DEVICE)
            fake = G(z)
            g_loss = -C(fake).mean()
            opt_g.zero_grad(set_to_none=True)
            g_loss.backward()
            opt_g.step()

            g_losses.append(g_loss.item()); c_losses.append(c_loss.item())

        hist["g_loss"].append(float(np.mean(g_losses)))
        hist["c_loss"].append(float(np.mean(c_losses)))
    return G, C, hist

wgan_G, wgan_C, wgan_hist = train_wgan_gp(Generator(Z_DIM), Critic(), train_loader, epochs=10)
def disable_inplace(module: nn.Module):
    for child in module.children():
        if isinstance(child, (nn.ReLU, nn.LeakyReLU)):
            child.inplace = False
        disable_inplace(child)

G = ConditionalGenerator(Z_DIM).to(DEVICE)
D = ConditionalDiscriminator().to(DEVICE)

disable_inplace(G)
disable_inplace(D)



WGAN-GP 1/10:   0%|          | 0/1490 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
def plot_hist(hist, title):
    plt.figure(figsize=(7, 4))
    for k, v in hist.items():
        plt.plot(v, label=k)
    plt.title(title)
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.grid(alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

plot_hist(gan_hist, "GAN learning curves")
plot_hist(cgan_hist, "cGAN learning curves")
plot_hist(wgan_hist, "WGAN-GP learning curves")


In [ ]:
summary = {
    "paths": {
        "RAW_IMAGES_DIR": str(RAW_IMAGES_DIR),
        "ATTR_CSV_PATH": str(ATTR_CSV_PATH),
        "PARTITION_CSV_PATH": str(PARTITION_CSV_PATH),
        "LANDMARKS_CSV_PATH": str(LANDMARKS_CSV_PATH),
        "PREPARED_IMAGES_DIR": str(PREPARED_IMAGES_DIR),
        "PREPARED_META_PATH": str(PREPARED_META_PATH),
    },
    "metrics": {
        "gan": gan_metrics,
        "cgan": cgan_metrics,
    },
}
pd.Series(summary).to_json(REPORTS_DIR / "lab7_summary.json", force_ascii=False, indent=2)
print("Saved:", REPORTS_DIR / "lab7_summary.json")
